In [1]:
!nvidia-smi
!ls -R /kaggle/input | head -20

Thu Sep 10 14:16:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
BASE = '/kaggle/input/datasets/bornamuzina/dataset448'
DATA = BASE + '/tensors_rgb_448_packed_k/tensors_rgb_448_packed'
!cp {BASE}/*.py /kaggle/working/
%cd /kaggle/working
!ls *.py

/kaggle/working
config.py  dataset.py  evaluate.py  targets.py	train.py


In [3]:
!apt-get install -qq python3.11 python3.11-venv python3.11-dev > /dev/null 2>&1
!python3.11 -m venv /kaggle/working/akv
!/kaggle/working/akv/bin/pip install -q --upgrade pip
!/kaggle/working/akv/bin/pip install -q akida-models==1.14.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 15.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
import re
src = open('config.py').read()
src = re.sub(r"^ROOT = Path\(.*?\)$", "ROOT = Path('/kaggle/working')", src, flags=re.M)
src = re.sub(r"^TENSOR_DIR = .*$", f"TENSOR_DIR = Path('{DATA}')", src, flags=re.M)
src = re.sub(r"^SPLITS_FILE = .*$", f"SPLITS_FILE = Path('{DATA}/splits.json')", src, flags=re.M)
src = re.sub(r"^RUNS_DIR = .*$", "RUNS_DIR = Path('/kaggle/working/runs')", src, flags=re.M)
open('config.py','w').write(src)

src = open('dataset.py').read()
src = src.replace(
    'if not npy.exists() or not meta_path.exists():',
    'npz = tensor_dir / f"{clip}_tensors.npz"\n    if not meta_path.exists() or (not npy.exists() and not npz.exists()):'
)
src = src.replace(
    'tensors = np.load(npy, mmap_mode="r")',
    'tensors = np.load(npy, mmap_mode="r") if npy.exists() else np.load(npz)["a"]'
)
open('dataset.py','w').write(src)

!grep -n "ROOT\|TENSOR_DIR\|SPLITS_FILE\|RUNS_DIR\|INPUT_SIZE\|^GRID" config.py

18:ROOT = Path('/kaggle/working')
19:TENSOR_DIR = Path('/kaggle/input/datasets/bornamuzina/dataset448/tensors_rgb_448_packed_k/tensors_rgb_448_packed')
20:# TENSOR_DIR = ROOT / "Data_new" / "tensors_messy"
21:# TENSOR_DIR = ROOT / "Data_new" / "tensors_rgb"
22:SPLITS_FILE = Path('/kaggle/input/datasets/bornamuzina/dataset448/tensors_rgb_448_packed_k/tensors_rgb_448_packed/splits.json')
23:RUNS_DIR = Path('/kaggle/working/runs')
30:# INPUT_SIZE = 224
31:INPUT_SIZE = 448
35:GRID = 14
36:CELL = INPUT_SIZE / GRID  # 32 px either way
41:SCALE = min(INPUT_SIZE / SRC_W, INPUT_SIZE / SRC_H)     # 0.70 at 448
42:PAD_X = (INPUT_SIZE - SRC_W * SCALE) / 2                # 0.0
43:PAD_Y = (INPUT_SIZE - SRC_H * SCALE) / 2                # 44.8


In [5]:
!MPLBACKEND=Agg /kaggle/working/akv/bin/python -u dataset.py

tensors : /kaggle/input/datasets/bornamuzina/dataset448/tensors_rgb_448_packed_k/tensors_rgb_448_packed
policy  : keep

TRAIN  (90 clips in split)
  clips loaded : 90
  samples      : 28045
  quiet frames : 0 (policy: keep)
  boxes        : 28345
  box size     : median 20.7 px (0.65 cells), p5 13.4, p95 35.4
  under 8 px   : 0.0%

VALIDATION  (12 clips in split)
  clips loaded : 12
  samples      : 3728
  quiet frames : 0 (policy: keep)
  boxes        : 3728
  box size     : median 20.4 px (0.64 cells), p5 13.0, p95 36.3
  under 8 px   : 0.0%

TEST  (12 clips in split)
  clips loaded : 12
  samples      : 3723
  quiet frames : 0 (policy: keep)
  boxes        : 4243
  box size     : median 21.2 px (0.66 cells), p5 16.2, p95 41.0
  under 8 px   : 0.0%

One batch:
  images shape : (4, 448, 448, 3)  float32
  value range  : 0.00 to 243.00
  zero pixels  : 20.1%
  boxes/frame  : [1, 1, 1, 1]
  out of bounds: 0
  inside padding: 0


In [6]:
!MPLBACKEND=Agg /kaggle/working/akv/bin/python -u train.py --epochs 15 --lr 1e-3 --batch_size 32 --name full_rgb_448

2026-09-10 14:20:13.594717: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1789050013.619113     238 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1789050013.626832     238 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1789050013.647810     238 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1789050013.647860     238 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1789050013.647866     238 computation_placer.cc:177] computation placer alr

In [7]:
!MPLBACKEND=Agg /kaggle/working/akv/bin/python -u evaluate.py --run full_rgb_448 --split validation

2026-09-10 16:51:08.459572: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1789059068.482595     352 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1789059068.490914     352 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1789059068.510176     352 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1789059068.510229     352 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1789059068.510237     352 computation_placer.cc:177] computation placer alr

In [8]:
!MPLBACKEND=Agg /kaggle/working/akv/bin/python -u evaluate.py --run full_rgb_448 --split test
!cd /kaggle/working && zip -qr full_rgb_448.zip runs/full_rgb_448
!ls -lh /kaggle/working/full_rgb_448.zip

2026-09-10 16:52:30.936671: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1789059150.960179    1342 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1789059150.967840    1342 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1789059150.988200    1342 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1789059150.988229    1342 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1789059150.988233    1342 computation_placer.cc:177] computation placer alr